In [6]:
from tensorflow.keras.models import load_model
import librosa
import soundfile as sf
import numpy as np
import joblib

In [7]:
def extract_mfcc(file_path, min_duration=2.0, max_frames=215, silence_threshold=-40):
    """Extract MFCCs and detect silence using RMS energy."""
    try:
        audio, sr = librosa.load(file_path, sr=22050)  # Ensure sample rate is consistent
        duration = librosa.get_duration(y=audio, sr=sr)

        segments = []
        if duration < min_duration:
            return None  # Skip very short audios
        else:
            num_segments = int(np.ceil(duration / 5))  # 5-second segments
            for i in range(num_segments):
                start = int(i * 5 * sr)
                end = int(start + 5 * sr)
                segment = audio[start:end]

                # Pad the last segment if needed
                if len(segment) < 5 * sr:
                    padding = int(5 * sr - len(segment))
                    segment = np.pad(segment, (0, padding))

                # Calculate RMS energy to detect silence
                rms = librosa.feature.rms(y=segment)
                avg_db = 20 * np.log10(np.mean(rms) + 1e-9)  # Avoid log(0)
                is_silent = avg_db < silence_threshold

                # Extract MFCCs only if not silent
                if not is_silent:
                    mfcc = librosa.feature.mfcc(y=segment, sr=sr, n_mfcc=13, hop_length=512)
                    if mfcc.shape[1] > max_frames:
                        mfcc = mfcc[:, :max_frames]
                    else:
                        mfcc = np.pad(mfcc, ((0, 0), (0, max_frames - mfcc.shape[1])), mode='constant')

                    mfcc = mfcc.T
                    segments.append(mfcc)
        return segments  # Returns only non-silent MFCC segments
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None

In [8]:
def predict_audio_rating(file_path):
    # Convert to WAV
    if not file_path.endswith(".wav"):
        SAMPLE_RATE = 22050
        y, sr = librosa.load(file_path, sr=SAMPLE_RATE)
        wav_path = "converted.wav"
        sf.write(wav_path, y, sr)
        file_path = wav_path

    model = load_model("Fluency and Pronunciation Rating model.keras", compile=False)
    scaler = joblib.load("mfcc_scaler.pkl")

    segments = extract_mfcc(file_path)
    if not segments:
        return "No usable speech found."

    X = np.array(segments)
    X_2d = X.reshape(-1, 13)
    X_scaled = scaler.transform(X_2d).reshape(len(X), 215, 13)

    pronun_preds, fluency_preds = model.predict(X_scaled)

    # Use top 25% scores
    top_n = max(1, int(0.25 * len(pronun_preds)))
    avg_pronun = np.mean(sorted(pronun_preds.flatten())[-top_n:])
    avg_fluency = np.mean(sorted(fluency_preds.flatten())[-top_n:])

    return {
        "Pronunciation Score": round(avg_pronun, 2),
        "Fluency Score": round(avg_fluency, 2)
    }


In [9]:
# Usage:
result = predict_audio_rating("motivation.mp3")
print(result)

C:\Users\rahul\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.5.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 51ms/step
{'Pronunciation Score': 7.98, 'Fluency Score': 9.12}


In [11]:
import librosa

def extract_pitch_energy(audio_file):
    y, sr = librosa.load(audio_file)
    pitches, magnitudes = librosa.piptrack(y=y, sr=sr)
    mean_pitch = pitches[pitches > 0].mean()
    energy = librosa.feature.rms(y=y).mean()
    return mean_pitch, energy

# Example
mean_pitch, energy = extract_pitch_energy('motivation.mp3')


In [12]:
mean_pitch, energy

(1395.782, 0.0914191)